In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
import tensorflow as tf
import numpy as np
import pandas as pd
import re
import string
# !pip install inltk
# from collections.abc import Iterable
from sklearn.feature_extraction.text import TfidfVectorizer
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Embedding,Bidirectional,Conv1D, MaxPooling1D, GlobalMaxPooling1D
from sklearn.model_selection import train_test_split

In [ ]:
!pip install --upgrade fastai
!pip install --upgrade inltk

!pip install abc
from inltk.inltk import setup

^C


In [4]:
# from inltk.inltk import setup

In [3]:
#Negative data
fake_csv=pd.read_csv("hindi_fake.csv")
#positive data
true_csv=pd.read_csv("hindi_true.csv")

In [4]:
fake_csv.head()
#fetches top 5 data from the data set
#for negative label is 0

,text,label
0,इसके पहले एक तस्वीर वायरल है जो रोड पर 'गो बैक...,0
1,धातु के पहियों वाले ट्रैक्टर्स की पुरानी और अस...,0
2,सोशल मीडिया पर वायरल एक पोस्ट में दावा किया जा...,0
3,उत्तर प्रदेश (Uttar Pradesh) में आत्महत्या (su...,0
4,दिल्ली की सीमाओं पर केंद्र द्वारा लागू तीन कृष...,0


In [5]:
true_csv.head()
# fetches top 5 data from data set
#for positive label is 1

,text,label
0,सात सालों तक एक दूसरे से नहीं की बातआपको जानकर...,1
1,"ममता बनर्जी की फोटो शेयर कर बोले प्रकाश राज, \...",1
2,"असम: कांग्रेस पर साधा राजनाथ सिंह ने निशाना, क...",1
3,सरवणन को टिकट मिलने से नाराजगीरविवार सुबह तिरु...,1
4,दिल'धक-धक गर्ल माधुरी दीक्षित के साथ आमिर ने स...,1


In [ ]:
true_csv.shape
#retuns the size of data set - being returns the number of rows and coloums

In [ ]:
fake_csv.shape

In [ ]:
#df=pd.concat([true_csv,fake_csv],axis=0)

In [ ]:
# Determine the size of the smaller dataset
min_size = min(len(true_csv), len(fake_csv))

# Undersample the larger dataset to match the size of the smaller dataset
# do chezz hota hai oversampling - isme data set ka size badadete hai
#undersampling - undersampling me bigger data set ko reduce krte hai - hum ye kiye hai
df_true_undersampled = true_csv.sample(n=min_size, random_state=42)
df_fake_undersampled = fake_csv.sample(n=min_size, random_state=42)
#since eak me 5177 hai or dusre me 3000 hai then hum extrat krte time same records extract kre.. isiye random state fix krte hai ..
# aur jankari ke liye gpt kre ....
# random_state = 42 fixes the randomness.
# When sharing findings or training models, reproducibility ensures consistency in results.

# Concatenate the undersampled datasets
df = pd.concat([df_true_undersampled, df_fake_undersampled], ignore_index=True)
# ingnore_index = true ka matlab hai ki nayi DataFrame ka index 0 se shuru hoke sequential hoga, purane indexes ko ignore karega.

# Shuffle the concatenated dataset
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
# The combined dataset df is shuffled to randomize the order of rows.
# frac=1 ensures that all rows are included in the shuffle.
# random_state=42 ensures reproducibility of the shuffle.
# reset_index(drop=True) resets the indices after shuffling and drops the old indices.


# Now df_equal_size contains both datasets of equal size

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
duplicate_rows = df[df.duplicated()]
print(duplicate_rows)
# is data set me duplicacy nai hai....
df = pd.concat([true_csv, fake_csv], axis=0).drop_duplicates()
#Tumhara df ab true_csv aur fake_csv ka combined version hai, jisme koi duplicate rows nahi hain. 😄
#Yeh check karta hai ki DataFrame df mein kaunsi rows duplicate (do baar ya usse zyada bar aayi hui) hain.


In [ ]:
# A pandas DataFrame is like a table of data, similar to a spreadsheet or an Excel file.
# It's a very powerful tool in Python for organizing, analyzing, and manipulating data. Think of it as a grid where:

# Rows: Represent individual records or entries.
# Columns: Represent different types of information about those entries.
# Key Features:
# Labeled Rows and Columns:

# Rows are labeled with an index (numbers by default).
# Columns have names (you can set them).
# Handles Different Types of Data:

# Each column can have a different type of data (numbers, text, dates, etc.).

# df.isnull():

# This creates a DataFrame of the same shape as df, where each cell contains:
# True if the value is missing (NaN).
# False if the value is not missing.

df.isnull().sum()
# Bas, ab tumhe pata chal gaya ki DataFrame mein kahaan-kahaan missing data hai!
# Har column ke True values ko add kar deta hai (True = 1, False = 0).
# Bas, ab tumhe pata chal gaya ki DataFrame mein kahaan-kahaan missing data hai!



In [ ]:
df = df.sample(frac = 1)
# frac=1 ka matlab hai ki pura DataFrame (100% rows) shuffle hoga

In [ ]:
erdf.head()

In [ ]:
(df['label'] == 0).sum()

In [ ]:
df

In [ ]:
#  wordopt is designed to clean and preprocess text by removing unwanted elements like special characters,
#  URLs, HTML tags, punctuation, and numbers
def wordopt(text):
    # text = text.lower()
    # Remove Text in Square Brackets:
    text = re.sub('\[.*?\]', '', text)
    text = re.sub("\\W"," ",text)
    text = re.sub('https?://\S+|www\.\S+', '', text)
    text = re.sub('<.*?>+', '', text)
    text = re.sub('[%s]' % re.escape(string.punctuation), '', text)
    text = re.sub('\n', '', text)
    text = re.sub('\w*\d\w*', '', text)
    return text

In [ ]:
# df_equal_size['text'] = df_equal_size['text'].apply(wordopt)
df['text'] = df['text'].apply(wordopt)


In [ ]:
X = df['text']
y= df['label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# Yeh line dataset ko train aur test sets mein split kar rahi hai
# Dataset ka 20% test set ke liye aur 80% train set ke liye rakha gaya hai.
# train_test_split() function sklearn library se aata hai aur dataset ko randomly train aur test parts mein todta hai.
# Ab tumhare paas alag-alag data hai training aur testing ke liye. 🎯
# X = df_equal_size['text']
# y= df_equal_size['label']
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Bhai, yeh code tumhare text data ko TF-IDF
#(Term Frequency-Inverse Document Frequency) ke vectors mein convert kar raha hai
# dekh bhai kya ho raha h na ki ki tfidf jo h woh text ko vector me convert kar deta h taki importance patha kar sake har text ki .. se
# "The cat sat on the mat."
# "The dog sat on the log."
# "The cat played with the dog."
# toh aab patha karna hoga na ki khon sa word ka importnace jadha h ,..toh interger me convert karna hoga tf-idf does that .. more frequest word like the, is, on get less
#tf-idf score since thier signifance is less ..so first ['cat', 'sat', 'on', 'mat', 'dog', 'log', 'played', 'with', 'the'] then
# WORD EMBEDDING
tfidf_vectorizer = TfidfVectorizer()
# ye line bus initialize kar di h tfidf ko
# tfidf_matrix = tfidf_vectorizer.fit_transform(df['text'])
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
# aab fit_tranform training data  ka tf-idf score nikal lega
# fit: TF-IDF model ko train karta hai X_train data ke basis pe (matlab vocabulary aur weights samajhta hai).
X_test_tfidf = tfidf_vectorizer.transform(X_test)
# aab ye transform bhi same kaam karega that is tfidf score hi nikalega matrix form me but here difference is ko test data ka hoga and ushi word ka nikalega jo trainign data me hoga
#The cat jumped":
# Document	cat	sat	on	mat	dog	log	played	with	the
# "The cat jumped."	0.7	0.0	0.0	0.0	0.0	0.0	0.0	0.0	0.0   .. jumped nahi h training data me toh yaha bhi nahi h .. see jumped ka score hi calcuate hi nahi hua h
# Output: Sparse matrix form mein vectors save hoti hain (memory-efficient representation).
# transform: X_train ko TF-IDF vectors mein convert karta hai.

In [ ]:
# Convert sparse matrices to dense arrays
X_train_dense = X_train_tfidf.toarray()
X_test_dense = X_test_tfidf.toarray()
# Sparse matrix ko dense array mein convert karta hai (Numpy array form)
# sparse matix - memory eff hota hai ..
# dense matrix islye chahiye kyuki ml algo dense favor krta hai
# sparcse matrix ko samjna muskil hota hai


In [ ]:
input_dim=X_train_dense.shape[0]
print(X_train_dense[1])
print(input_dim)

# X_train_dense.shape()
#It gives the dimensions of the array as a tuple: (number_of_rows, number_of_columns)
# Document	cat	sat	on	mat	dog	log	played	with	the
# "The cat sat on the mat."	0.4	0.4	0.4	0.5	0.0	0.0	0.0	0.0	0.0
# "The dog sat on the log."	0.0	0.4	0.4	0.0	0.4	0.5	0.0	0.0	0.0
# "The cat played with the dog."	0.3	0.0	0.0	0.0	0.3	0.0	0.4	0.4	0.0
# so we can see ki 6926 jo h woh total no of statement h .. like  the cat sat ..the dog sat ..sara add karke and 181119 jo columnn aaya tha woh total no of column h


In [ ]:
# first model ann

In [ ]:
from keras.models import Sequential
from keras.layers import Dense, Embedding, Flatten
from keras.optimizers import Adam
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import numpy as np

# Initialize the model
model = Sequential()

# Embedding layer: Maps each word index to a dense vector representation
model.add(Embedding(input_dim=X_train_dense.shape[1], output_dim=128, input_length=X_train_dense.shape[1]))

# Flatten layer: Converts 2D embedding outputs into a 1D vector for input to Dense layers
model.add(Flatten())

# Dense layer: Fully connected layer with ReLU activation
model.add(Dense(units=128, activation='relu'))

# Output layer: Single neuron with sigmoid activation for binary classification
model.add(Dense(units=1, activation='sigmoid'))

# Compile the model: Binary crossentropy for binary classification and Adam optimizer
model.compile(loss='binary_crossentropy', optimizer=Adam(learning_rate=0.001), metrics=['accuracy'])

# Train the model: Fit on training data
history = model.fit(X_train_dense, y_train, epochs=10, batch_size=32, validation_data=(X_test_dense, y_test))

# Evaluate the model on test data
loss, accuracy = model.evaluate(X_test_dense, y_test)
print(f'Test Loss: {loss}, Test Accuracy: {accuracy}')

# Predict on test data
y_pred = (model.predict(X_test_dense) > 0.5).astype(int)

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Class 0', 'Class 1'])
disp.plot(cmap=plt.cm.Blues)
plt.title("Confusion Matrix")
plt.show()


In [ ]:
#2nd model rnn

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Get predictions from the model
predictions = model.predict(X_test_dense)

# Convert probabilities to binary predictions
binary_predictions = (predictions > 0.5).astype(int)

# Compute confusion matrix
cm = confusion_matrix(y_test, binary_predictions)

# Extract the counts
true_negatives = cm[0, 0]
false_positives = cm[0, 1]
false_negatives = cm[1, 0]
true_positives = cm[1, 1]

# Print actual vs predicted results
print(f"Actual Positive: {true_positives + false_negatives}")
print(f"Actual Negative: {true_negatives + false_positives}")
print(f"Predicted Positive: {true_positives + false_positives}")
print(f"Predicted Negative: {true_negatives + false_negatives}")

# Display confusion matrix
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Negative", "Positive"])
disp.plot(cmap="Blues")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
!pip install matplotlib scikit-learn

from keras.models import Sequential
from keras.layers import Embedding, SimpleRNN, Dense
from keras.optimizers import Adam
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import numpy as np

# Initialize the model
model = Sequential()

# Embedding layer: Maps each word index to a dense vector representation
model.add(Embedding(input_dim=X_train_dense.shape[1], output_dim=128, input_length=X_train_dense.shape[1]))

# RNN layer: Processes the sequence step by step and learns dependencies
model.add(SimpleRNN(units=128, activation='tanh', return_sequences=False))

# Output layer: Single neuron with sigmoid activation for binary classification
model.add(Dense(units=1, activation='sigmoid'))

# Compile the model: Binary crossentropy for binary classification and Adam optimizer
model.compile(loss='binary_crossentropy', optimizer=Adam(learning_rate=0.001), metrics=['accuracy'])

# Train the model: Fit on training data
history = model.fit(X_train_dense, y_train, epochs=10, batch_size=32, validation_data=(X_test_dense, y_test))

# Evaluate the model on test data
loss, accuracy = model.evaluate(X_test_dense, y_test)
print(f'Test Loss: {loss}, Test Accuracy: {accuracy}')

# Predict on test data
y_pred = (model.predict(X_test_dense) > 0.5).astype(int)

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Class 0', 'Class 1'])
disp.plot(cmap=plt.cm.Blues)
plt.title("Confusion Matrix")
plt.show()


In [ ]:
# LSTM/BiLSTM model: Accuracy: 59.87%

In [ ]:
import tensorflow as TF
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import numpy as np p

# Define the Bidirectional LSTM model
model = Sequential()

# Add Embedding layer
model.add(Embedding(input_dim=X_train_dense.shape[1], output_dim=128, input_length=X_train_dense.shape[1]))

# Add Bidirectional LSTM layer
model.add(Bidirectional(LSTM(units=128)))

# Add Dense layer with sigmoid activation
model.add(Dense(1, activation='sigmoid'))

# Compile the model
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

# Train the model and store the training history
history = model.fit(
    X_train_dense, y_train,
    epochs=10, batch_size=32,
    validation_data=(X_test_dense, y_test)
)

# Evaluate the model
loss, accuracy = model.evaluate(X_test_dense, y_test)
print(f'Test Loss: {loss}, Test Accuracy: {accuracy}')

# Plot training and validation accuracy
train_accuracy = history.history['accuracy']
val_accuracy = history.history['val_accuracy']
epochs = range(1, len(train_accuracy) + 1)

plt.figure(figsize=(8, 6))
plt.plot(epochs, train_accuracy, 'bo-', label='Training Accuracy')
plt.plot(epochs, val_accuracy, 'ro-', label='Validation Accuracy')
plt.title('Training and Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)
plt.show()

# Generate predictions
y_pred = model.predict(X_test_dense)
y_pred_labels = (y_pred > 0.5).astype(int)  # Convert probabilities to binary predictions

# Generate confusion matrix
cm = confusion_matrix(y_test, y_pred_labels)

# Plot confusion matrix
plt.figure(figsize=(8, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Negative", "Positive"])
disp.plot(cmap="Blues", values_format='d')
plt.title('Confusion Matrix')
plt.show()


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense
import matplotlib.pyplot as plt

# Define the Bidirectional LSTM model
model = Sequential()

# Add Embedding layer
model.add(Embedding(input_dim=X_train_dense.shape[1], output_dim=128, input_length=X_train_dense.shape[1]))

# Add Bidirectional LSTM layer
model.add(Bidirectional(LSTM(units=128)))

# Add Dense layer with sigmoid activation
model.add(Dense(1, activation='sigmoid'))

# Compile the model
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

# Train the model and store the training history
history = model.fit(
    X_train_dense, y_train,
    epochs=10, batch_size=32,
    validation_data=(X_test_dense, y_test)
)

# Evaluate the model
loss, accuracy = model.evaluate(X_test_dense, y_test)
print(f'Test Loss: {loss}, Test Accuracy: {accuracy}')

# Plot training and validation accuracy
train_accuracy = history.history['accuracy']
val_accuracy = history.history['val_accuracy']
epochs = range(1, len(train_accuracy) + 1)

plt.figure(figsize=(8, 6))
plt.plot(epochs, train_accuracy, 'bo-', label='Training Accuracy')
plt.plot(epochs, val_accuracy, 'ro-', label='Validation Accuracy')
plt.title('Training and Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
model = Sequential()
# Creates an empty, sequential model container where layers will be added step-by-step.
model.add(Embedding(input_dim=X_train_dense.shape[1], output_dim=128, input_length=X_train_dense.shape[1]))

model.add(Bidirectional(LSTM(units=128)))
# Add a Bidirectional LSTM layer
# : For the sentence "I love ice cream," a Bidirectional LSTM looks at:

# Forward pass: "I → love → ice → cream"
# Backward pass: "cream → ice → love → I"
model.add(Dense(1, activation='sigmoid'))
# Add a Dense (fully connected) layer
#  Adds a fully connected output layer.
# Dense(1): There is one output neuron since this is a binary classification task (e.g., predicting "positive" or "negative").
# activation='sigmoid': The sigmoid activation function outputs probabilities between 0 and 1, which is ideal for binary classification
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
# binary_crossentropy measures how far the predicted probabilities are from the true labels (0 or 1).
# Example: If the true label is 1 (positive sentiment) and the model predicts 0.8, the loss will be small. If the model predicts 0.1, the loss will be larger.
# adam adjusts the model weights during training to minimize the loss. It’s fast and widely used.
model.fit(X_train_dense, y_train, epochs=10, batch_size=32, validation_data=(X_test_dense, y_test))
# X_train_dense: Training input (e.g., sequences of numbers representing words).
# y_train: Training labels (e.g., 0 for negative, 1 for positive).
# epochs=10: The model sees the entire training data 10 times.
# batch_size=32: Processes 32 examples at a time instead of the whole dataset (memory-efficient).
# Suppose X_train_dense contains sequences like [[1, 2, 3], [4, 5, 6]], and y_train contains [1, 0] (positive/negative labels). During training:

# The model makes predictions for [[1, 2, 3]] and compares them to 1.
# It calculates the loss (how wrong the prediction is).
# The optimizer (adam) adjusts weights to improve future predictions.
# This repeats for all sequences in batches of 32.

# Evaluate model
loss, accuracy = model.evaluate(X_test_dense, y_test)
print(f'Test Loss: {loss}, Test Accuracy: {accuracy}')

In [ ]:
model.summary()

In [ ]:
#CNN Model: accuracy: 60.89%

In [ ]:
# model = Sequential()
# model.add(Embedding(input_dim=X_train_dense.shape[1], output_dim=128, input_length=X_train_dense.shape[1]))
# model.add(Conv1D(filters=128, kernel_size=5, activation='relu'))
# model.add(MaxPooling1D(pool_size=2))
# model.add(Conv1D(filters=128, kernel_size=5, activation='relu'))
# model.add(GlobalMaxPooling1D())
# model.add(Dense(128, activation='relu'))
# model.add(Dense(1, activation='sigmoid'))

# # Compile model
# model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

# # Train model
# model.fit(X_train_dense, y_train, epochs=10, batch_size=32, validation_data=(X_test_dense, y_test))

# # Evaluate model
# loss, accuracy = model.evaluate(X_test_dense, y_test)
# print(f'Test Loss: {loss}, Test Accuracy: {accuracy}')

import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, MaxPooling1D, GlobalMaxPooling1D, Dense

# Define the model
model = Sequential()
model.add(Embedding(input_dim=X_train_dense.shape[1], output_dim=128, input_length=X_train_dense.shape[1]))
model.add(Conv1D(filters=128, kernel_size=5, activation='relu'))
model.add(MaxPooling1D(pool_size=2))
model.add(Conv1D(filters=128, kernel_size=5, activation='relu'))
model.add(GlobalMaxPooling1D())
model.add(Dense(128, activation='relu'))
model.add(Dense(1, activation='sigmoid'))

# Compile the model
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

# Train the model and store the history
history = model.fit(X_train_dense, y_train, epochs=10, batch_size=32, validation_data=(X_test_dense, y_test))

# Plot the loss graph
plt.figure(figsize=(10, 6))
plt.plot(history.history['loss'], label='Training Loss', marker='o', linestyle='-')
plt.plot(history.history['val_loss'], label='Validation Loss', marker='s', linestyle='--')
plt.title('Training and Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, MaxPooling1D, GlobalMaxPooling1D, Dense
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import numpy as np

# Define the model
model = Sequential()
model.add(Embedding(input_dim=X_train_dense.shape[1], output_dim=128, input_length=X_train_dense.shape[1]))
model.add(Conv1D(filters=128, kernel_size=5, activation='relu'))
model.add(MaxPooling1D(pool_size=2))
model.add(Conv1D(filters=128, kernel_size=5, activation='relu'))
model.add(GlobalMaxPooling1D())
model.add(Dense(128, activation='relu'))
model.add(Dense(1, activation='sigmoid'))

# Compile the model
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

# Train the model and store the history
history = model.fit(X_train_dense, y_train, epochs=10, batch_size=32, validation_data=(X_test_dense, y_test))

# Plot the loss graph
plt.figure(figsize=(10, 6))
plt.plot(history.history['loss'], label='Training Loss', marker='o', linestyle='-')
plt.plot(history.history['val_loss'], label='Validation Loss', marker='s', linestyle='--')
plt.title('Training and Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid()
plt.show()

# Evaluate the model
loss, accuracy = model.evaluate(X_test_dense, y_test)
print(f'Test Loss: {loss}, Test Accuracy: {accuracy}')

# Predict on test data
y_pred = (model.predict(X_test_dense) > 0.5).astype(int)

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Class 0', 'Class 1'])
disp.plot(cmap=plt.cm.Blues)
plt.title("Confusion Matrix")
plt.show()


In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt

# Define the LSTM model
def create_lstm_model(input_shape):
    model = tf.keras.Sequential()
    model.add(tf.keras.layers.Input(shape=input_shape))
    model.add(tf.keras.layers.LSTM(units=128))
    model.add(tf.keras.layers.Dense(128, activation='relu'))
    model.add(tf.keras.layers.Dense(1, activation='sigmoid'))
    model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model

# Assuming you already have X_train_embeddings, y_train, X_test_embeddings, y_test loaded
input_shape = (X_train_embeddings.shape[1], X_train_embeddings.shape[2])
lstm_model = create_lstm_model(input_shape)

# Display the model summary
lstm_model.summary()

# Train the model and capture the history
history = lstm_model.fit(
    X_train_embeddings, y_train,
    epochs=10, batch_size=32,
    validation_data=(X_test_embeddings, y_test)
)

# Evaluate the model
lstm_model.evaluate(X_test_embeddings, y_test)

# Plot training and validation accuracy
train_accuracy = history.history['accuracy']
val_accuracy = history.history['val_accuracy']
epochs = range(1, len(train_accuracy) + 1)

plt.figure(figsize=(8, 6))
plt.plot(epochs, train_accuracy, 'bo-', label='Training Accuracy')
plt.plot(epochs, val_accuracy, 'ro-', label='Validation Accuracy')
plt.title('Training and Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
## cnn + lstm: accuracy=59%

In [ ]:
model = Sequential()
model.add(Embedding(input_dim=X_train_dense.shape[1], output_dim=128, input_length=X_train_dense.shape[1]))
model.add(Conv1D(filters=128, kernel_size=5, activation='relu'))
model.add(MaxPooling1D(pool_size=2))
model.add(LSTM(units=128))
model.add(Dense(128, activation='relu'))
model.add(Dense(1, activation='sigmoid'))

# Compile model
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

# Train model
model.fit(X_train_dense, y_train, epochs=10, batch_size=32, validation_data=(X_test_dense, y_test))

# Evaluate model
loss, accuracy = model.evaluate(X_test_dense, y_test)
print(f'Test Loss: {loss}, Test Accuracy: {accuracy}')

In [ ]:
!pip install fasttext


In [ ]:
# import fasttext
!pip install fasttest
!pip install fasttext


In [ ]:
# # Split the combined DataFrame into training and testing sets
# train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

# # Separate the text and label data for training and testing
# X_train = train_df['text']
# y_train = train_df['label']
# X_test = test_df['text']
# y_test = test_df['label']

In [ ]:
# print("Data type of y_train:", type(y_train))
# y_train_list = y_train.tolist()


In [ ]:
# # Prepare training data in FastText format
# with open('train.txt', 'w') as f:
#     for i, text in enumerate(X_train_dense):
#         if i < len(y_train):  # Check if the index is within the bounds of y_train
#             label = '__label__' + str(y_train_list[i])
#             f.write(label + ' ' + ' '.join(map(str, text)) + '\n')
#         else:
#             print(f"Warning: Missing label for sample at index {i}")

# # Train the FastText model
# model = fasttext.train_supervised('train.txt', epoch=10, lr=1.0)

# # Prepare test data in FastText format
# with open('test.txt', 'w') as f:
#     for i, text in enumerate(X_test_dense):
#         f.write(' '.join(map(str, text)) + '\n')

# # Evaluate the model
# results = model.test('test.txt')
# print(f'Test Accuracy: {results[1]:.4f}')

In [ ]:
print(len(y_train))

In [ ]:
print(len(X_train_dense))

In [ ]:
# FASTTEXT MODEL: accuracy=93%
# import fasttext



# def wordopt(text):
#     text = re.sub('\[.*?\]', '', text)
#     text = re.sub("\\W", " ", text)
#     text = re.sub('https?://\S+|www\.\S+', '', text)
#     text = re.sub('<.*?>+', '', text)
#     text = re.sub('[%s]' % re.escape(string.punctuation), '', text)
#     text = re.sub('\n', '', text)
#     text = re.sub('\w*\d\w*', '', text)
#     return text

# df['text'] = df['text'].apply(wordopt)


# X = df['text']
# y = df['label']
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# # Convert labels to the FastText format
# def format_label(label):
#     return '__label__' + str(label)

# # Create training data with labels in FastText format
# y_train_list = y_train.tolist()
# formatted_labels_train = [format_label(label) for label in y_train_list]

# # Write the training data to a file
# with open('train.txt', 'w') as f:
#     for text, label in zip(X_train, formatted_labels_train):
#         f.write(label + ' ' + text + '\n')

# # Create test data with labels in FastText format
# y_test_list = y_test.tolist()
# formatted_labels_test = [format_label(label) for label in y_test_list]

# # Write the test data to a file
# with open('test.txt', 'w') as f:
#     for text, label in zip(X_test, formatted_labels_test):
#         f.write(label + ' ' + text + '\n')

# # Train the FastText model
# model = fasttext.train_supervised(input='train.txt')

# # Evaluate the model using FastText's built-in method
# test_result = model.test('test.txt')
# print(f'Number of examples: {test_result[0]}')
# print(f'Precision: {test_result[1]}')
# print(f'Recall: {test_result[2]}')

In [ ]:
#fastext-tokenization
import re
import string
import fasttext

def wordopt(text):
    text = re.sub('\[.*?\]', '', text)
    text = re.sub("\\W", " ", text)
    text = re.sub('https?://\S+|www\.\S+', '', text)
    text = re.sub('<.*?>+', '', text)
    text = re.sub('[%s]' % re.escape(string.punctuation), '', text)
    text = re.sub('\n', '', text)
    text = re.sub('\w*\d\w*', '', text)
    return text

df['text'] = df['text'].apply(wordopt)

X = df['text']
y = df['label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train a FastText model on the preprocessed text
train_file = 'train_ft.txt'
with open(train_file, 'w') as f:
    for text, label in zip(X_train, y_train):
        f.write(f'__label__{label} {text}\n')

fasttext_model = fasttext.train_supervised(input=train_file)

# Define a function to get FastText embeddings
def get_fasttext_embeddings(texts, model, max_length=128):
    embeddings = []
    for text in texts:
        words = text.split()
        word_embeddings = [model.get_word_vector(word) for word in words]
        if len(word_embeddings) < max_length:
            # Pad with zeros if shorter than max_length
            word_embeddings += [np.zeros(model.get_dimension())] * (max_length - len(word_embeddings))
        else:
            # Truncate if longer than max_length
            word_embeddings = word_embeddings[:max_length]
        embeddings.append(word_embeddings)
    return np.array(embeddings)

X_train_embeddings = get_fasttext_embeddings(X_train, fasttext_model)
X_test_embeddings = get_fasttext_embeddings(X_test, fasttext_model)


In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt

# Define the LSTM model
def create_lstm_model(input_shape):
    model = tf.keras.Sequential()
    model.add(tf.keras.layers.Input(shape=input_shape))
    model.add(tf.keras.layers.LSTM(units=128))
    model.add(tf.keras.layers.Dense(128, activation='relu'))
    model.add(tf.keras.layers.Dense(1, activation='sigmoid'))
    model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model

# Assuming you already have X_train_embeddings, y_train, X_test_embeddings, y_test loaded
input_shape = (X_train_embeddings.shape[1], X_train_embeddings.shape[2])
lstm_model = create_lstm_model(input_shape)

# Display the model summary
lstm_model.summary()

# Train the model and capture the history
history = lstm_model.fit(
    X_train_embeddings, y_train,
    epochs=10, batch_size=32,
    validation_data=(X_test_embeddings, y_test)
)

# Evaluate the model
lstm_model.evaluate(X_test_embeddings, y_test)

# Plot training and validation accuracy
train_accuracy = history.history['accuracy']
val_accuracy = history.history['val_accuracy']
epochs = range(1, len(train_accuracy) + 1)

plt.figure(figsize=(8, 6))
plt.plot(epochs, train_accuracy, 'bo-', label='Training Accuracy')
plt.plot(epochs, val_accuracy, 'ro-', label='Validation Accuracy')
plt.title('Training and Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
#cnn 62%
def create_cnn_model(input_shape):
    model = tf.keras.Sequential()
    model.add(tf.keras.layers.Input(shape=input_shape))
    model.add(tf.keras.layers.Conv1D(filters=128, kernel_size=5, activation='relu'))
    model.add(tf.keras.layers.MaxPooling1D(pool_size=2))
    model.add(tf.keras.layers.Dense(128, activation='relu'))
    model.add(tf.keras.layers.Dense(1, activation='sigmoid'))
    model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model

cnn_model = create_cnn_model((X_train_embeddings.shape[1], X_train_embeddings.shape[2]))
cnn_model.summary()
cnn_model.fit(X_train_embeddings, y_train, epochs=10, batch_size=32, validation_data=(X_test_embeddings, y_test))
cnn_model.evaluate(X_test_embeddings, y_test)


In [ ]:
#lstm 89.2%
def create_lstm_model(input_shape):
    model = tf.keras.Sequential()
    model.add(tf.keras.layers.Input(shape=input_shape))
    model.add(tf.keras.layers.LSTM(units=128))
    model.add(tf.keras.layers.Dense(128, activation='relu'))
    model.add(tf.keras.layers.Dense(1, activation='sigmoid'))
    model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model

lstm_model = create_lstm_model((X_train_embeddings.shape[1], X_train_embeddings.shape[2]))
lstm_model.summary()
lstm_model.fit(X_train_embeddings, y_train, epochs=10, batch_size=32, validation_data=(X_test_embeddings, y_test))
lstm_model.evaluate(X_test_embeddings, y_test)


In [ ]:
#cnn-lstm 89%
def create_cnn_lstm_model(input_shape):
    model = tf.keras.Sequential()
    model.add(tf.keras.layers.Input(shape=input_shape))
    model.add(tf.keras.layers.Conv1D(filters=128, kernel_size=5, activation='relu'))
    model.add(tf.keras.layers.MaxPooling1D(pool_size=2))
    model.add(tf.keras.layers.LSTM(units=128))
    model.add(tf.keras.layers.Dense(128, activation='relu'))
    model.add(tf.keras.layers.Dense(1, activation='sigmoid'))
    model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model

cnn_lstm_model = create_cnn_lstm_model((X_train_embeddings.shape[1], X_train_embeddings.shape[2]))
cnn_lstm_model.summary()
cnn_lstm_model.fit(X_train_embeddings, y_train, epochs=10, batch_size=32, validation_data=(X_test_embeddings, y_test))
cnn_lstm_model.evaluate(X_test_embeddings, y_test)


In [ ]:
pip install transformers datasets tensorflow


In [ ]:
from transformers import BertTokenizer, TFBertForSequenceClassification
from transformers import DataCollatorWithPadding
from datasets import Dataset

In [ ]:
#muril:87%

In [ ]:
# import numpy as np
# import tensorflow as tf
# from transformers import AutoTokenizer, TFAutoModel

# # Load the MURIL tokenizer and model
# model_name = "google/muril-base-cased"
# tokenizer = AutoTokenizer.from_pretrained(model_name)
# muril_model = TFAutoModel.from_pretrained(model_name)



# # Convert target variables to NumPy arrays
# y_train = np.array(y_train)
# y_test = np.array(y_test)

# def tokenize_and_encode(texts, tokenizer, max_length=128):
#     input_ids = []
#     attention_masks = []

#     for text in texts:
#         encoded = tokenizer.encode_plus(
#             text,
#             add_special_tokens=True,
#             max_length=max_length,
#             padding='max_length',  # Ensure padding to max_length
#             truncation=True,       # Ensure truncation to max_length
#             return_attention_mask=True,
#             return_tensors='tf'
#         )

#         input_ids.append(encoded['input_ids'])
#         attention_masks.append(encoded['attention_mask'])

#     return tf.concat(input_ids, axis=0), tf.concat(attention_masks, axis=0)

# # Tokenize and encode the training data
# X_train_encodings, X_train_masks = tokenize_and_encode(X_train, tokenizer)
# X_test_encodings, X_test_masks = tokenize_and_encode(X_test, tokenizer)

# # Extract embeddings from MURIL
# def extract_embeddings(bert_model, input_ids, attention_masks):
#     outputs = bert_model(input_ids, attention_mask=attention_masks)
#     embeddings = outputs.last_hidden_state
#     return embeddings

# # Define a function to process in smaller batches
# def batch_process(func, bert_model, input_ids, attention_masks, batch_size=2):
#     results = []
#     for i in range(0, input_ids.shape[0], batch_size):
#         batch_input_ids = input_ids[i:i + batch_size]
#         batch_attention_masks = attention_masks[i:i + batch_size]
#         result = func(bert_model, batch_input_ids, batch_attention_masks)
#         results.append(result)
#     return tf.concat(results, axis=0)

# X_train_embeddings = batch_process(extract_embeddings, muril_model, X_train_encodings, X_train_masks)
# X_test_embeddings = batch_process(extract_embeddings, muril_model, X_test_encodings, X_test_masks)

# # Free up memory by deleting the input encodings and masks
# del X_train_encodings, X_train_masks, X_test_encodings, X_test_masks

# # Define your CNN/LSTM/BiLSTM model
# model = tf.keras.Sequential()
# model.add(tf.keras.layers.Input(shape=(X_train_embeddings.shape[1], X_train_embeddings.shape[2])))
# model.add(tf.keras.layers.Conv1D(filters=128, kernel_size=5, activation='relu'))
# model.add(tf.keras.layers.MaxPooling1D(pool_size=2))
# model.add(tf.keras.layers.LSTM(units=128))
# model.add(tf.keras.layers.Dense(128, activation='relu'))
# model.add(tf.keras.layers.Dense(1, activation='sigmoid'))

# # Compile model with mixed precision policy
# policy = tf.keras.mixed_precision.Policy('mixed_float16')
# tf.keras.mixed_precision.set_global_policy(policy)

# model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

# # Train model with reduced batch size
# model.fit(X_train_embeddings, y_train, epochs=10, batch_size=2, validation_data=(X_test_embeddings, y_test))

# # Evaluate model
# loss, accuracy = model.evaluate(X_test_embeddings, y_test)
# print(f'Test Loss: {loss}, Test Accuracy: {accuracy}')


In [ ]:
import numpy as np
import tensorflow as tf
from transformers import AutoTokenizer, TFAutoModel

# Load the MURIL tokenizer and model
model_name = "google/muril-base-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
muril_model = TFAutoModel.from_pretrained(model_name)

def tokenize_and_encode(texts, tokenizer, max_length=128):
    input_ids = []
    attention_masks = []

    for text in texts:
        encoded = tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=max_length,
            padding='max_length',  # Ensure padding to max_length
            truncation=True,       # Ensure truncation to max_length
            return_attention_mask=True,
            return_tensors='tf'
        )

        input_ids.append(encoded['input_ids'])
        attention_masks.append(encoded['attention_mask'])

    return tf.concat(input_ids, axis=0), tf.concat(attention_masks, axis=0)

# Tokenize and encode the training and testing data
X_train_encodings, X_train_masks = tokenize_and_encode(X_train, tokenizer)
X_test_encodings, X_test_masks = tokenize_and_encode(X_test, tokenizer)


In [ ]:
def extract_embeddings(bert_model, input_ids, attention_masks):
    outputs = bert_model(input_ids, attention_mask=attention_masks)
    embeddings = outputs.last_hidden_state
    return embeddings

# Define a function to process in smaller batches
def batch_process(func, bert_model, input_ids, attention_masks, batch_size=2):
    results = []
    for i in range(0, input_ids.shape[0], batch_size):
        batch_input_ids = input_ids[i:i + batch_size]
        batch_attention_masks = attention_masks[i:i + batch_size]
        result = func(bert_model, batch_input_ids, batch_attention_masks)
        results.append(result)
    return tf.concat(results, axis=0)

X_train_embeddings = batch_process(extract_embeddings, muril_model, X_train_encodings, X_train_masks)
X_test_embeddings = batch_process(extract_embeddings, muril_model, X_test_encodings, X_test_masks)

# Free up memory by deleting the input encodings and masks
del X_train_encodings, X_train_masks, X_test_encodings, X_test_masks


In [ ]:
#82.7%
def create_cnn_model(input_shape):
    model = tf.keras.Sequential()
    model.add(tf.keras.layers.Input(shape=input_shape))
    model.add(tf.keras.layers.Conv1D(filters=128, kernel_size=5, activation='relu'))
    model.add(tf.keras.layers.MaxPooling1D(pool_size=2))
    model.add(tf.keras.layers.Dense(128, activation='relu'))
    model.add(tf.keras.layers.Dense(1, activation='sigmoid'))
    model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model

cnn_model = create_cnn_model((X_train_embeddings.shape[1], X_train_embeddings.shape[2]))
cnn_model.summary()
cnn_model.fit(X_train_embeddings, y_train, epochs=10, batch_size=2, validation_data=(X_test_embeddings, y_test))
cnn_model.evaluate(X_test_embeddings, y_test)


In [ ]:
def create_lstm_model(input_shape):
    model = tf.keras.Sequential()
    model.add(tf.keras.layers.Input(shape=input_shape))
    model.add(tf.keras.layers.LSTM(units=128))
    model.add(tf.keras.layers.Dense(128, activation='relu'))
    model.add(tf.keras.layers.Dense(1, activation='sigmoid'))
    model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model

lstm_model = create_lstm_model((X_train_embeddings.shape[1], X_train_embeddings.shape[2]))
lstm_model.summary()
lstm_model.fit(X_train_embeddings, y_train, epochs=10, batch_size=2, validation_data=(X_test_embeddings, y_test))
lstm_model.evaluate(X_test_embeddings, y_test)


In [ ]:
def create_bilstm_model(input_shape):
    model = tf.keras.Sequential()
    model.add(tf.keras.layers.Input(shape=input_shape))
    model.add(tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(units=128)))
    model.add(tf.keras.layers.Dense(128, activation='relu'))
    model.add(tf.keras.layers.Dense(1, activation='sigmoid'))
    model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model

bilstm_model = create_bilstm_model((X_train_embeddings.shape[1], X_train_embeddings.shape[2]))
bilstm_model.summary()
bilstm_model.fit(X_train_embeddings, y_train, epochs=10, batch_size=2, validation_data=(X_test_embeddings, y_test))
bilstm_model.evaluate(X_test_embeddings, y_test)


In [ ]:
def create_cnn_bilstm_model(input_shape):
    model = tf.keras.Sequential()
    model.add(tf.keras.layers.Input(shape=input_shape))
    model.add(tf.keras.layers.Conv1D(filters=128, kernel_size=5, activation='relu'))
    model.add(tf.keras.layers.MaxPooling1D(pool_size=2))
    model.add(tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(units=128)))
    model.add(tf.keras.layers.Dense(128, activation='relu'))
    model.add(tf.keras.layers.Dense(1, activation='sigmoid'))
    model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model

cnn_bilstm_model = create_cnn_bilstm_model((X_train_embeddings.shape[1], X_train_embeddings.shape[2]))
cnn_bilstm_model.summary()
cnn_bilstm_model.fit(X_train_embeddings, y_train, epochs=10, batch_size=2, validation_data=(X_test_embeddings, y_test))
cnn_bilstm_model.evaluate(X_test_embeddings, y_test)


In [ ]:
#bert:92.6%

In [ ]:
from transformers import BertTokenizer

# Load the BERT tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-multilingual-cased')

# Tokenize the text data
def tokenize_text(texts, tokenizer, max_len=128):
    input_ids = []
    attention_masks = []

    for text in texts:
        encoded_dict = tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=max_len,
            padding='max_length',
            return_attention_mask=True,
            return_tensors='tf',
            truncation=True
        )

        input_ids.append(encoded_dict['input_ids'])
        attention_masks.append(encoded_dict['attention_mask'])

    return tf.concat(input_ids, axis=0), tf.concat(attention_masks, axis=0)

X_train_tokens, X_train_masks = tokenize_text(X_train, tokenizer)
X_test_tokens, X_test_masks = tokenize_text(X_test, tokenizer)


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv1D, GlobalMaxPooling1D, Dense, Dropout

def create_cnn_model(input_shape):
    input_ids = Input(shape=input_shape, dtype='int32')
    attention_masks = Input(shape=input_shape, dtype='int32')

    embedding_layer = tf.keras.layers.Embedding(input_dim=tokenizer.vocab_size, output_dim=128)(input_ids)

    conv_layer = Conv1D(filters=128, kernel_size=5, activation='relu')(embedding_layer)
    pooling_layer = GlobalMaxPooling1D()(conv_layer)

    dropout_layer = Dropout(0.5)(pooling_layer)
    output_layer = Dense(1, activation='sigmoid')(dropout_layer)

    model = Model(inputs=[input_ids, attention_masks], outputs=output_layer)
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

    return model

cnn_model = create_cnn_model((128,))
cnn_model.summary()
cnn_model.fit([X_train_tokens, X_train_masks], y_train, batch_size=32, epochs=3, validation_split=0.2)
cnn_model.evaluate([X_test_tokens, X_test_masks], y_test)

In [ ]:
#lstm
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout, Embedding

def create_lstm_model(input_shape):
    input_ids = Input(shape=input_shape, dtype='int32')
    attention_masks = Input(shape=input_shape, dtype='int32')

    embedding_layer = Embedding(input_dim=tokenizer.vocab_size, output_dim=128)(input_ids)

    lstm_layer = LSTM(128)(embedding_layer)

    dropout_layer = Dropout(0.5)(lstm_layer)
    output_layer = Dense(1, activation='sigmoid')(dropout_layer)

    model = Model(inputs=[input_ids, attention_masks], outputs=output_layer)
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

    return model

lstm_model = create_lstm_model((128,))
lstm_model.summary()
lstm_model.fit([X_train_tokens, X_train_masks], y_train, batch_size=32, epochs=3, validation_split=0.2)
lstm_model.evaluate([X_test_tokens, X_test_masks], y_test)


In [ ]:
#cnn-lstm
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv1D, LSTM, GlobalMaxPooling1D, Dense, Dropout, Embedding

def create_cnn_lstm_model(input_shape):
    input_ids = Input(shape=input_shape, dtype='int32')
    attention_masks = Input(shape=input_shape, dtype='int32')

    embedding_layer = Embedding(input_dim=tokenizer.vocab_size, output_dim=128)(input_ids)

    conv_layer = Conv1D(filters=128, kernel_size=5, activation='relu')(embedding_layer)
    lstm_layer = LSTM(128)(conv_layer)

    dropout_layer = Dropout(0.5)(lstm_layer)
    output_layer = Dense(1, activation='sigmoid')(dropout_layer)

    model = Model(inputs=[input_ids, attention_masks], outputs=output_layer)
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

    return model

cnn_lstm_model = create_cnn_lstm_model((128,))
cnn_lstm_model.summary()
cnn_lstm_model.fit([X_train_tokens, X_train_masks], y_train, batch_size=32, epochs=3, validation_split=0.2)
cnn_lstm_model.evaluate([X_test_tokens, X_test_masks], y_test)
